In [1]:
"""
Phase 1 - Step 1: Feature Engineering
--------------------------------------
Your project brief asks for these input features:
  cycle count, avg discharge temp, depth of discharge, charge rate,
  time since last full charge

The cleaned CSV already has: Cycle_Index, Ambient_Temperature/Max_Temp,
Voltage_Drop_Rate, Internal_Resistance. This script adds the two that
are still missing: Depth_of_Discharge and a Charge_Rate proxy.
"""
import pandas as pd
import numpy as np

df = pd.read_csv("battery_dataset_clean.csv")

# -------------------------------------------------------------
# FEATURE 1: Depth of Discharge (DoD) - NOT AVAILABLE, documented honestly
# Your project brief asks for DoD as an input. However, NASA's test
# protocol always discharges each cycle fully down to the SAME fixed
# cutoff voltage (2.7V) - every cycle is a ~100% depth discharge by
# design. There is no cycle-to-cycle DoD variation in this dataset,
# so this feature would be constant (useless) or fabricated data
# leakage if approximated from capacity. We DO NOT include DoD here -
# this is an honest dataset limitation to note in your report.
# (Real EV telemetry, unlike this lab dataset, WOULD have varying DoD,
# since drivers rarely drain a pack to 0% each day - worth flagging
# as a gap between this lab data and real-world EV behavior.)
# -------------------------------------------------------------

# -------------------------------------------------------------
# FEATURE 2: Charge Rate proxy (C-rate-like signal)
# Meaning: how "hard"/fast the battery was drained. We don't have
# a direct current value in this per-cycle summary table, so we
# approximate it using Voltage_Drop_Rate - a battery drained faster
# under a heavier load tends to lose voltage more quickly per second.
# (This is a proxy feature - documented honestly as such, not a
# true C-rate measurement, since raw current isn't in our summary table.)
# -------------------------------------------------------------
df["Charge_Rate_Proxy"] = df["Voltage_Drop_Rate_V_per_sec"] * 1000  # scaled for readability

# -------------------------------------------------------------
# FEATURE 3: Time since last full charge (proxy)
# Meaning: elapsed cycles since the last impedance/reset event as
# a stand-in for "how long it's been running without a full reset."
# We already track Cycles_Since_Impedance - reuse it directly,
# renamed for clarity in the model's feature list.
# -------------------------------------------------------------
df["Time_Since_Reset_Cycles"] = df["Cycles_Since_Impedance"]

# Save enriched dataset
df.to_csv("battery_dataset_features.csv", index=False)

print("New features added. Preview:")
print(df[["Battery_ID", "Discharge_Index", "Charge_Rate_Proxy",
          "Time_Since_Reset_Cycles", "SoH"]].head(10))
print(f"\nSaved -> battery_dataset_features.csv  ({len(df)} rows)")

New features added. Preview:
  Battery_ID  Discharge_Index  Charge_Rate_Proxy  Time_Since_Reset_Cycles  \
0      B0005                1           0.427893                        0   
1      B0005                2           0.436387                        0   
2      B0005                3           0.420707                        0   
3      B0005                4           0.439346                        0   
4      B0005                5           0.452136                        0   
5      B0005                6           0.456665                        0   
6      B0005                7           0.465775                        0   
7      B0005                8           0.426520                        0   
8      B0005                9           0.446452                        0   
9      B0005               10           0.460787                        0   

        SoH  
0  1.000000  
1  0.994527  
2  0.988614  
3  0.988567  
4  0.988235  
5  0.988782  
6  0.988504  
7  0.983447